《随机森林构建测试版》重点：非正式！！！

In [1]:
import numpy as np
import pandas as pd
import RNA
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import LeaveOneOut

1. 定义特征提取函数

In [4]:
def extract_features(spacer, target, mismatch_pos):
    # 1. Spacer GC%
    gc = (spacer.count("G") + spacer.count("C")) / len(spacer) * 100

    # 2. Spacer 自身 MFE
    fc_s = RNA.fold_compound(spacer)
    _, mfe_spacer = fc_s.mfe()

    # 3. Spacer-Target 杂交结合能 (Delta G duplex) - 修正 API 接口
    duplex = RNA.duplexfold(spacer, target)
    dg_duplex = duplex.energy

    # 4. 位置权重惩罚评分 (种子区 1-8 nt 权重更大)
    if mismatch_pos > 0:
        weight = 2.0 if mismatch_pos <= 8 else 1.0
    else:
        weight = 0.0

    return [gc, mfe_spacer, dg_duplex, mismatch_pos, weight]

2. 模拟构建数据

In [5]:
# df = pd.read_csv("cas12a_training_data.csv")
data = [
    {
        "spacer": "UAGCACCAUCUGAAAUCGGU",
        "target": "TAGCACCATCTGAAATCGGT",
        "pos": 0,
        "activity": 1.0,
    },
    {
        "spacer": "UAGCACCAUCUGAAAUCGGU",
        "target": "TAGCACCATTTGAAATCGGT",
        "pos": 10,
        "activity": 0.15,
    },
    {
        "spacer": "UAGCACCAUCUGAAAUCGGU",
        "target": "TAGCACCATCTGAAAACGGT",
        "pos": 16,
        "activity": 0.60,
    },
]
df = pd.DataFrame(data)

# 生成特征矩阵 X 和标签 y
X = np.array([
    extract_features(row["spacer"], row["target"], row["pos"])
    for _, row in df.iterrows()
])
y = df["activity"].values

3. 随机森林建模 + LOOCV 评估

In [6]:
loo = LeaveOneOut()
y_true, y_pred = [], []

rf = RandomForestRegressor(n_estimators=50, random_state=42, max_depth=3)

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)

    y_true.append(y_test[0])
    y_pred.append(pred[0])

# 计算评估指标
rho, _ = spearmanr(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)

print(f"=== 随机森林 LOOCV 评估结果 ===")
print(f"Spearman 秩相关系数 (r_s): {rho:.3f}")
print(f"均方误差 (MSE): {mse:.4f}")

=== 随机森林 LOOCV 评估结果 ===
Spearman 秩相关系数 (r_s): -1.000
均方误差 (MSE): 0.3384
